In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import matplotlib.ticker as ticker
from matplotlib import rcParams
import matplotlib.colors as mcolors
import os
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

# ==========================================
# === 0. User-controllable parameters (Vertical line and text positions) ===
# ==========================================
ANNOT_POS_CONFIG = {
    0: {"offset_x": 0.5, "y_ratio": 0.85}, 
    2: {"offset_x": 0.5, "y_ratio": 0.85}, 
    4: {"offset_x": 0.5, "y_ratio": 0.85}, 
    8: {"offset_x": 0.5, "y_ratio": 0.85}, 
}
persist_hours = 0.0      # 0.0 = No persistence required (earliest intersection); change to 0.5 or 1.0 to avoid transient crossings

# Core modification: Global fixed parameter changed to gamma_ja
GAMMA_JA_FIXED = 1.05

# ==========================================
# === 0.1 Global Size and Layout Control ===
# ==========================================
# Canvas size
FIG_WIDTH = 10         # Canvas width
FIG_HEIGHT = 7.5       # Canvas height

# Font sizes
FONT_TITLE = 30        # Main title font size
FONT_LABEL = 30        # X/Y axis label font size (e.g., "Time (h)")
FONT_TICK = 26         # X/Y axis tick number font size
FONT_OFFSET = 26       # Y axis top-left scientific notation font size
FONT_LEGEND = 18       # Legend font size
LEGEND_BBOX_X = 0.99   # Legend horizontal position
LEGEND_BBOX_Y = 1.015  # Legend vertical position
FONT_INSET_TICK = 24   # Inset plot tick number font size
FONT_INSET_OFFSET = 22 # Inset plot scientific notation font size
FONT_INSET_TEXT = 20   # Inset plot text font size (next to dashed line, t=...)

# Line and tick control
LINE_AXES = 2.0        # Outer border and tick mark thickness
TICK_LENGTH_MAIN = 6.0 # Main plot tick line length
TICK_LENGTH_INSET = 4.0# Inset plot tick line length
LINE_PLOT_MAIN = 2.2   # Main plot growth curve thickness
LINE_PLOT_INSET = 2.0  # Inset plot growth curve thickness
LINE_VLINE = 1.8       # Inset plot vertical dashed intersection line thickness
LINE_ZOOM_BOX = 1.5    # Main plot gray indicator box thickness

# Padding control
PAD_TITLE = 20         # Distance between title and top of the frame
PAD_YLABEL = 15        # Distance between Y-axis label and Y-axis numbers

# ==========================================
# === 1. Global Plot Style Settings ===
# ==========================================
config = {
    "font.family": 'sans-serif',
    "font.sans-serif": ['Arial'],
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "mathtext.fontset": 'custom',
    "mathtext.rm": 'Arial',
    "mathtext.it": 'Arial:italic',
    "mathtext.bf": 'Arial:bold',
    "axes.linewidth": LINE_AXES,
    "xtick.major.width": LINE_AXES,
    "ytick.major.width": LINE_AXES,
    "xtick.major.size": TICK_LENGTH_MAIN,    # Apply main plot X-axis tick length
    "ytick.major.size": TICK_LENGTH_MAIN,    # Apply main plot Y-axis tick length
    "xtick.direction": 'out',
    "ytick.direction": 'out',
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "axes.labelsize": FONT_LABEL,
    "figure.figsize": (FIG_WIDTH, FIG_HEIGHT),
    "axes.unicode_minus": False
}
rcParams.update(config)

# ==========================================
# === 2. Parameter Initialization ===
# ==========================================
np.random.seed(42)

num_S = np.random.randint(13, 29)
num_R = np.random.randint(2, 4)

print(f"Current community complexity: Sensitive strains (S) = {num_S}, Resistant strains (R) = {num_R}")

S0_total = 1e7
R0_total = 1e6
S0 = (S0_total / num_S) * np.ones(num_S)
R0 = (R0_total / num_R) * np.ones(num_R)

K_cap = 1e9
d = 0.07

# Asymmetric interspecific competition
common_intra = np.random.uniform(0.95, 1.05)
gamma_ia = common_intra
gamma_jb = common_intra

DELTA_MIN, DELTA_MAX = 0.02, 0.12
GAMMA_IB_CAP = 1.40
EPS = 1e-6

# Core modification: Fixed gamma_ja
gamma_ja = float(GAMMA_JA_FIXED)
delta = float(np.random.uniform(DELTA_MIN, DELTA_MAX))
gamma_ib = min(gamma_ja + delta, GAMMA_IB_CAP)
if gamma_ib <= gamma_ja:
    gamma_ib = gamma_ja + EPS

print(f"Competition parameters: gamma_ja (R inhibits S) = {gamma_ja:.2f}, gamma_ib (S inhibits R) = {gamma_ib:.2f}")

Vsmax = np.random.uniform(0.98, 1.0, num_S)
Vrmax = np.random.uniform(0.89, 0.90, num_R)
Vsmin = -0.1
Vrmin = -0.1

# Core modification: Restore random distribution for Ks, keep random distribution for MICs
MICs = np.random.uniform(2.0, 8.0, num_S)         
Ks = np.random.uniform(4.0, 6.0, num_S)        

# Resistant strains remain random (consistent with original logic)
MICr = np.random.uniform(64.0, 256.0, num_R)
Kr = np.random.uniform(1.0, 2.0, num_R)

total_time = 72
c_values = [0, 2, 4, 8]
# Use 5000 precision uniformly to ensure accurate intersection capture
t_span = np.linspace(0, total_time, 5000)

# ==========================================
# === 3. Define ODE Equations ===
# ==========================================
def bacteria_growth(y, t, c):
    S = y[:num_S]
    R = y[num_S:]
    total_S = np.sum(S)
    total_R = np.sum(R)

    comp_s = 1 - (gamma_ia * total_S + gamma_ja * total_R) / K_cap
    comp_r = 1 - (gamma_ib * total_S + gamma_jb * total_R) / K_cap

    Vs_rate = Vsmax - ((Vsmax - Vsmin) * (c / MICs) ** Ks) / ((c / MICs) ** Ks - (Vsmin / Vsmax))
    Vr_rate = Vrmax - ((Vrmax - Vrmin) * (c / MICr) ** Kr) / ((c / MICr) ** Kr - (Vrmin / Vrmax))

    dSdt = Vs_rate * S * comp_s - d * S
    dRdt = Vr_rate * R * comp_r - d * R
    return np.concatenate([dSdt, dRdt])

# ==========================================
# === 4. Color Mapping (Lancet Style) ===
# ==========================================
# R: Red gradient
lancet_red_light = '#F6B3B3'
lancet_red_deep  = '#ED0000'
cmap_red = mcolors.LinearSegmentedColormap.from_list("LancetRedGrad", [lancet_red_light, lancet_red_deep])
colors_r = cmap_red(np.linspace(0.25, 1.0, num_R))

# S: Blue gradient
lancet_blue_light = '#ADB6C4'
lancet_blue_deep  = '#00468B'
cmap_lancet = mcolors.LinearSegmentedColormap.from_list("LancetGrad", [lancet_blue_light, lancet_blue_deep])
colors_s = cmap_lancet(np.linspace(0.2, 1.0, num_S))

# ==========================================
# === 5. Set Save Path ===
# ==========================================
# Use relative path for GitHub compatibility
output_dir = "./results_dynamics"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# ==========================================
# === 6. Loop Simulation and Plotting ===
# ==========================================
for c in c_values:
    y0 = np.concatenate([S0, R0])
    solution = odeint(bacteria_growth, y0, t_span, args=(c,))

    # =========================================================
    # Use [Total Abundance] to calculate intersection t_cross and y_cross
    # =========================================================
    dt_span = t_span[1] - t_span[0]
    n_persist = max(1, int(persist_hours / dt_span)) 

    total_S_curve = np.sum(solution[:, :num_S], axis=1)
    total_R_curve = np.sum(solution[:, num_S:], axis=1)
    diff = total_R_curve - total_S_curve

    idx = np.where((diff[:-1] <= 0) & (diff[1:] > 0))[0] + 1
    t_cross, y_cross = None, None

    for i in idx:
        if n_persist > 1:
            if i + n_persist > len(diff) or not np.all(diff[i:i + n_persist] > 0):
                continue

        t1, t2 = t_span[i - 1], t_span[i]
        d1, d2 = diff[i - 1], diff[i]
        if d2 == d1: continue
        
        # Calculate exact time t_cross
        t_cross = t1 - d1 * (t2 - t1) / (d2 - d1)
        
        # Calculate intersection height y_cross
        y1, y2 = total_R_curve[i - 1], total_R_curve[i]
        y_cross = y1 + (y2 - y1) * (t_cross - t1) / (t2 - t1)
        break

    if t_cross is not None:
        print(f"c={c}: Found [Total Abundance] intersection at t={t_cross:.2f}h, height y={y_cross:.2e}")
    else:
        print(f"c={c}: No reversal occurred")

    # Dynamic Y-axis
    current_data_max = np.max(solution)
    y_top_limit = max(100, current_data_max * 1.15)
    if y_top_limit > K_cap:
        y_top_limit = K_cap * 1.2

    fig, ax = plt.subplots()

    # --- Main plot (Apply global linewidth) ---
    for k in range(num_S):
        ax.plot(t_span, solution[:, k], color=colors_s[k], linewidth=LINE_PLOT_MAIN, zorder=5)
    for k in range(num_R):
        ax.plot(t_span, solution[:, num_S + k], color=colors_r[k], linewidth=LINE_PLOT_MAIN, zorder=10)

    # Draw hidden lines to provide legends
    ax.plot([], [], color=lancet_blue_deep, linewidth=LINE_PLOT_MAIN, label='Sensitive')
    ax.plot([], [], color=lancet_red_deep, linewidth=LINE_PLOT_MAIN, label='Resistant')

    ax.set_xlim(0, total_time)
    ax.set_xticks(np.arange(0, total_time + 1, 12))
    ax.set_ylim(0, y_top_limit)
    ax.set_xlabel("Time (h)")
    # Apply dynamic padding
    ax.set_ylabel("Species abundance", labelpad=PAD_YLABEL)

    # Apply dynamic font, padding, and \gamma_ja formula format to title
    ax.set_title(rf"$\gamma_{{ja}}$ = {GAMMA_JA_FIXED}, $c$ = {c}", fontsize=FONT_TITLE, pad=PAD_TITLE)

    # Deploy legend
    leg = ax.legend(loc='upper right', bbox_to_anchor=(LEGEND_BBOX_X, LEGEND_BBOX_Y), fontsize=FONT_LEGEND, frameon=False)
    leg.set_zorder(100)

    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 0))
    ax.yaxis.set_major_formatter(formatter)
    ax.yaxis.get_offset_text().set_fontsize(FONT_OFFSET)

    # =========================================================
    # Dynamically adjust the range of the local inset zoom
    # =========================================================
    if t_cross is not None and y_cross is not None:
        zoom_t_max = max(12.0, np.ceil(t_cross) + 3.0) 
        
        target_zoom_y = y_cross * 1.5   
        min_zoom_y = S0_total * 2.0  
        zoom_y_max = max(min_zoom_y, target_zoom_y)
    else:
        zoom_t_max = 12.0
        zoom_y_max = S0_total * 5.0 

    # --- Local inset zoom (Apply global linewidth) ---
    axins = ax.inset_axes([0.58, 0.32, 0.38, 0.38], zorder=30)

    for k in range(num_S):
        axins.plot(t_span, solution[:, k], color=colors_s[k], linewidth=LINE_PLOT_INSET, zorder=5)
    for k in range(num_R):
        axins.plot(t_span, solution[:, num_S + k], color=colors_r[k], linewidth=LINE_PLOT_INSET, zorder=10)

    axins.set_xlim(0, zoom_t_max)
    axins.set_ylim(0, zoom_y_max)
    
    # Dynamically set X-axis ticks of the inset to prevent crowding
    if zoom_t_max <= 15:
        axins.set_xticks(np.arange(0, zoom_t_max + 1, 2))
    elif zoom_t_max <= 30:
        axins.set_xticks(np.arange(0, zoom_t_max + 1, 5))
    else:
        axins.set_xticks(np.arange(0, zoom_t_max + 1, 10))

    # ==============================================================================
    # Draw dashed lines and hanging text (no arrows, no background box)
    # ==============================================================================
    if t_cross is not None:
        cfg = ANNOT_POS_CONFIG.get(c, {"offset_x": 0.4, "y_ratio": 0.85})
        cur_offset_x = cfg["offset_x"]
        cur_y_ratio = cfg["y_ratio"]

        # Draw the penetrating dashed line
        axins.axvline(x=t_cross, color='black', linestyle='--', linewidth=LINE_VLINE, zorder=20)

        text_x = t_cross + cur_offset_x
        text_y = zoom_y_max * cur_y_ratio
        align_ha = 'left' if cur_offset_x >= 0 else 'right'

        # Anti-collision mechanism: if text is too far right and about to overflow, automatically snap it to the left of the dashed line
        if text_x > zoom_t_max * 0.9:
             text_x = t_cross - abs(cur_offset_x) - 0.1
             align_ha = 'right'

        axins.text(
            text_x, text_y,
            rf'$\hat{{t}} = {t_cross:.2f}$',
            ha=align_ha, va='center',
            family='Arial', fontsize=FONT_INSET_TEXT, fontweight='bold', color='black',
            zorder=40
        )
    # ==============================================================================

    # Apply specific tick length, font size, and thickness for the inset zoom
    axins.tick_params(axis='both', labelsize=FONT_INSET_TICK, length=TICK_LENGTH_INSET, width=LINE_AXES)
    formatter_ins = ticker.ScalarFormatter(useMathText=True)
    formatter_ins.set_powerlimits((0, 0))
    axins.yaxis.set_major_formatter(formatter_ins)
    axins.yaxis.get_offset_text().set_fontsize(FONT_INSET_OFFSET)

    ax.indicate_inset_zoom(axins, edgecolor="dimgray", alpha=0.6, linewidth=LINE_ZOOM_BOX, zorder=25)

    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.tick_params(top=False, right=False, which='both')
    
    plt.tight_layout()

    # === Save ===
    filename = f'Lancet_GammaJA_{GAMMA_JA_FIXED}_C_{c}.png'
    full_save_path = os.path.join(output_dir, filename)
    plt.savefig(full_save_path, dpi=600, bbox_inches='tight')
    print(f"Saved: {full_save_path}")
    
    # Free memory
    plt.close()